In [3]:
import torch
import torch as ch
import torch.nn.functional as F

In [5]:
def project_onto_set(z, constraint_set):
    """
    Project z onto the constraint set K.
    For simplicity, assume K is a Euclidean ball of radius R.
    """
    R = constraint_set['radius']
    norm = torch.norm(z)
    if norm > R:
        z = z * (R / norm)
    return z

def estimate_gradient(W, x, y, constraint_set, m=10, gamma=ch.Tensor([[0.1]]), sigma=1.0):
    """
    Algorithm 1: Approximate Stochastic Gradient Estimation for Unknown-Index Case
    """
    k, d = W.shape  # k models, d-dimensional input
    mu = W @ x  # Compute outputs of all k models
    
    # Line 5: Initialize z^(0) as the projection of 0 onto K
    z = torch.zeros_like(mu)
    z = project_onto_set(z, constraint_set)
    
    # Lines 6-9: Perform m stochastic updates
    for t in range(m):
        # Line 7: Sample ξ from N(0, I)
        xi = torch.randn_like(z)
        
        # Line 8: Update z^(t)
        z = z - (gamma / (2 * sigma**2)) * (z - mu) + torch.sqrt(gamma) * xi
        z = project_onto_set(z, constraint_set)
    
    # Lines 10-12: Compute gradient components g_j
    g = torch.zeros_like(W)
    for j in range(k):
        # For the max self-selection criterion, the gradient depends on whether
        # the j-th model's output is the maximum
        if mu[j] == torch.max(mu):
            g[j] = (1 / sigma**2) * (y - mu[j]) * x
        else:
            g[j] = (1 / sigma**2) * (z[j] - mu[j]) * x
    
    # Line 13: Return the gradient vector
    return g

def projected_sgd(X, Y, k, constraint_set, T=100, lambda_=1.0):
    """
    Algorithm 2: Projected Stochastic Gradient Descent for Unknown-Index Case
    """
    n_samples, d = X.shape
    W = torch.zeros(k, d, requires_grad=False)  # Initialize model parameters
    
    # Lines 3-7: Perform T iterations of PSGD
    for t in range(1, T + 1):
        # Line 4: Set learning rate η_t
        eta_t = 1 / (lambda_ * t)
        
        # Line 5: Estimate gradient using Algorithm 1
        # For simplicity, process one sample at a time
        i = torch.randint(0, n_samples, (1,)).item()
        x = X[i]
        y = Y[i]
        g = estimate_gradient(W, x, y, constraint_set)
        
        # Line 6: Update W^(t) using projected gradient step
        for j in range(k):
            W[j] = project_onto_set(W[j] - eta_t * g[j], constraint_set)
    
    # Line 8: Return the final model parameters
    return W

# Example usage: Treatment Effect Estimation
if __name__ == "__main__":
    # Generate synthetic data
    n_samples = 10000
    n_features = 5  # Features: e.g., age, education level, income, etc.
    k = 2  # Two models: treatment and control
    
    # True model parameters
    true_W = torch.tensor([
        [1.5, -0.5, 2.0, 0.5, -1.0],  # Treatment model (w1)
        [1.0, 0.0, 1.5, -0.5, 0.5]    # Control model (w2)
    ])
    
    # Generate features (X) and outcomes (Y)
    X = torch.randn(n_samples, n_features)
    Y1 = true_W[0] @ X.T  # Outcome under treatment
    Y2 = true_W[1] @ X.T  # Outcome under control
    Y = torch.maximum(Y1, Y2)  # Observed outcome (max self-selection)
    
    # Define constraint set (e.g., Euclidean ball with radius R)
    constraint_set = {'radius': 2.0}
    
    # Run projected SGD
    W_final = projected_sgd(X, Y, k, constraint_set, T=1000, lambda_=1.0)
    
    # Print results
    print("Final model parameters (W):")
    print(W_final)
    print("True model parameters (true_W):")
    print(true_W)

Final model parameters (W):
tensor([[-1.2121,  0.3372, -1.5442,  0.0677,  0.1666],
        [-1.2121,  0.3372, -1.5442,  0.0677,  0.1666]])
True model parameters (true_W):
tensor([[ 1.5000, -0.5000,  2.0000,  0.5000, -1.0000],
        [ 1.0000,  0.0000,  1.5000, -0.5000,  0.5000]])
